# Extended Properties Viewer - Session-Scope Analysis

**Minimal example of querying session-level extended properties and visualizing distributions.**

This notebook demonstrates:
- Using the high-level QueryBackend API (same as dashboard widgets)
- Querying session-scope extended properties
- Visualizing property distributions with seaborn violin plots

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from collab_env.dashboard.widgets.analysis_context import AnalysisContext
from collab_env.dashboard.widgets.query_scope import QueryScope, ScopeType
from collab_env.data.db.query_backend import QueryBackend

## Setup: Initialize QueryBackend and Session Scope

### Recommended: Connect to Cloud SQL via Auth Proxy

**Google's recommended approach** for connecting to Cloud SQL is using the Cloud SQL Auth Proxy, which provides:
- ✅ Automatic SSL/TLS encryption
- ✅ IAM-based authentication
- ✅ No need to manage authorized networks
- ✅ Works with both public and private IP

**Setup (one-time):**
```bash
# Terminal 1: Start Cloud SQL Auth Proxy
./cloud-sql-proxy PROJECT_ID:REGION:INSTANCE_NAME --port 5433

# Keep this running while using the notebook
```

**In the notebook:** Set environment variables to connect via proxy on localhost:

```python
import os
os.environ['DB_BACKEND'] = 'postgres'
os.environ['POSTGRES_HOST'] = 'localhost'
os.environ['POSTGRES_PORT'] = '5433'  # Proxy port
os.environ['POSTGRES_DB'] = 'tracking_analytics'
os.environ['POSTGRES_USER'] = 'postgres'
os.environ['POSTGRES_PASSWORD'] = 'your-password'  # Or fetch from Secret Manager
```

Then skip to **Cell 8** to use `QueryBackend()` with environment variables.

---

### Alternative (Not Recommended): Direct Public IP Connection

⚠️ **Warning**: Direct connections bypass the proxy's security features. Google recommends using the Auth Proxy instead.

If you must connect directly (e.g., testing, special circumstances):

In [ ]:
# RECOMMENDED: Configure environment for Cloud SQL Auth Proxy connection
# Run this cell if you have the proxy running in another terminal

import os
import subprocess

# Fetch password from Secret Manager (recommended)


def get_password_from_secret_manager(secret_name="postgres-password"):
    """Fetch password from Google Cloud Secret Manager."""
    try:
        result = subprocess.run(
            [
                "gcloud",
                "secrets",
                "versions",
                "access",
                "latest",
                "--secret",
                secret_name,
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"⚠ Failed to fetch secret: {e}")
        return None


# Configure environment variables for proxy connection
os.environ["DB_BACKEND"] = "postgres"
os.environ["POSTGRES_HOST"] = "localhost"  # Connect via proxy on localhost
os.environ["POSTGRES_PORT"] = (
    "5433"  # Proxy port (use 5433 if local postgres uses 5432)
)
os.environ["POSTGRES_DB"] = "tracking_analytics"
os.environ["POSTGRES_USER"] = "postgres"
os.environ["POSTGRES_PASSWORD"] = (
    get_password_from_secret_manager() or "your-password-here"
)

print("✓ Environment configured for Cloud SQL Auth Proxy connection")
print(f"  Proxy endpoint: localhost:{os.environ['POSTGRES_PORT']}")
print(f"  Database: {os.environ['POSTGRES_DB']}")
print(f"  User: {os.environ['POSTGRES_USER']}")

# Now skip to Cell 8 to initialize QueryBackend() with these settings

---

**If using Cloud SQL Auth Proxy (recommended), skip cells 6-9 below and jump directly to Cell 10.**

---

In [2]:
# Option 1: Fetch password from Google Cloud Secret Manager (recommended)
# Requires: gcloud CLI installed and authenticated
import subprocess


def get_password_from_secret_manager(secret_name="postgres-password"):
    """Fetch password from Google Cloud Secret Manager."""
    try:
        result = subprocess.run(
            [
                "gcloud",
                "secrets",
                "versions",
                "access",
                "latest",
                "--secret",
                secret_name,
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"⚠ Failed to fetch secret: {e}")
        return None


# Option 2: Specify password directly (not recommended for shared notebooks)
# db_password = 'your-password-here'


# Try Secret Manager first, fallback to environment variable
db_password = get_password_from_secret_manager() or os.getenv("POSTGRES_PASSWORD")

if db_password:
    print("✓ Password retrieved successfully")
else:
    print("⚠ No password found - will fail to connect")

✓ Password retrieved successfully


In [3]:
# Construct database configuration for direct Cloud SQL connection
from collab_env.data.db.config import DBConfig, PostgresConfig

# Cloud SQL instance details (modify with your values)
CLOUD_SQL_PUBLIC_IP = "34.67.80.127"  # Get from: gcloud sql instances describe INSTANCE_NAME --format="value(ipAddresses[0].ipAddress)"
DB_NAME = "tracking_analytics"
DB_USER = "postgres"
DB_PORT = 5432  # Default PostgreSQL port

# Create PostgresConfig with direct connection parameters
postgres_config = PostgresConfig(
    host=CLOUD_SQL_PUBLIC_IP,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=db_password,
)

# Create DBConfig with postgres backend
db_config = DBConfig(backend="postgres")
db_config.postgres = postgres_config  # Override with custom config

print(f"✓ Database configuration created:")
print(f"  Host: {postgres_config.host}")
print(f"  Database: {postgres_config.dbname}")
print(f"  User: {postgres_config.user}")
print(f"  Password set: {postgres_config.password is not None}")

✓ Database configuration created:
  Host: 34.67.80.127
  Database: tracking_analytics
  User: postgres
  Password set: True


In [4]:
# Initialize QueryBackend with custom Cloud SQL config
query_backend = QueryBackend(config=db_config)
print("✓ QueryBackend initialized with direct Cloud SQL connection")

# Test connection by listing sessions
try:
    sessions = query_backend.get_sessions()
    print(f"\n✓ Connection successful! Found {len(sessions)} sessions")
except Exception as e:
    print(f"\n✗ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Verify Cloud SQL public IP is correct")
    print("  2. Ensure your IP is in authorized networks:")
    print(
        "     gcloud sql instances patch INSTANCE_NAME --authorized-networks=YOUR_IP/32"
    )
    print("  3. Check password is correct")
    print("  4. Verify database exists and user has permissions")

OperationalError: (psycopg2.OperationalError) connection to server at "34.67.80.127", port 5432 failed: Operation timed out
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

**Notes on Direct Public IP Connections:**

⚠️ **Google Cloud Recommendation**: Use the Cloud SQL Auth Proxy instead of direct connections for better security.

If using direct connections:
1. **Security**: Requires SSL/TLS configuration and authorized networks management
2. **Public IP**: Get your instance's public IP with:
   ```bash
   gcloud sql instances describe INSTANCE_NAME --format="value(ipAddresses[0].ipAddress)"
   ```
3. **Firewall**: Must authorize your IP address:
   ```bash
   gcloud sql instances patch INSTANCE_NAME --authorized-networks=YOUR_IP/32
   ```
4. **SSL**: Should configure SSL certificates for encrypted connections

**Recommended**: See [docs/dashboard/CLOUD_SETUP.md](../dashboard/CLOUD_SETUP.md) for Cloud SQL Auth Proxy setup.

---

**If cells 3-6 above worked, skip cell 8 below and proceed directly to cell 9.**

---

### Default: Connect Using Environment Variables (Recommended with Proxy)

In [ ]:
# Initialize QueryBackend (reads from environment variables)
# Set DUCKDB_PATH or DB_BACKEND as needed before running
# Example: os.environ['DUCKDB_PATH'] = '/path/to/your.duckdb'

query_backend = QueryBackend()
print("✓ QueryBackend initialized")

In [ ]:
# List available sessions
sessions = query_backend.get_sessions("boids_2d_rollout")
print(f"Found {len(sessions)} sessions:")
sessions[["session_id"]]

In [ ]:
# Select a session to analyze (modify as needed)
session_id = (
    "rollout-boid_food_basic_vpluspplus_a_n0_h1_vr0.1_s0_rollout5_selfloops-unknown"
)

## Query Extended Properties at Session Scope

This uses the same high-level API as the dashboard's `ExtendedPropertiesViewerWidget`.

In [ ]:
# Create session-scope QueryScope
scope = QueryScope.from_session(
    session_id=session_id, agent_type="agent"  # Filter to 'agent' type (not 'target')
)

# Create AnalysisContext (mimics dashboard widget pattern)
context = AnalysisContext(query_backend=query_backend, scope=scope)

print(f"✓ Created context with scope: {scope}")

In [ ]:
# Get available properties for this session
params = context.get_query_params()
available_props = query_backend.get_available_properties(**params)

print(f"\nAvailable properties ({len(available_props)} total):")
available_props[["property_id", "property_name", "data_type", "unit"]]

In [ ]:
# Query property distributions (aggregated across all episodes in session)
distributions = query_backend.get_property_distributions(**params)

print(f"\n✓ Loaded {len(distributions)} property observations")
print(f"  Properties: {distributions['property_id'].unique().tolist()}")

In [ ]:
distributions.head()

## Visualize Selected Properties with Violin Plots

In [ ]:
# Select properties to visualize (modify as needed)
# Example: attention weight properties from GNN models
selected_properties = ["attn_weight_food", "attn_weight_boid", "attn_weight_self"]

# Filter distributions to selected properties
filtered_df = distributions[
    distributions["property_id"].isin(selected_properties)
].copy()

if len(filtered_df) == 0:
    print(f"⚠ No data found for properties: {selected_properties}")
    print(f"   Available properties: {distributions['property_id'].unique().tolist()}")
else:
    print(
        f"✓ Filtered to {len(filtered_df)} observations across {len(selected_properties)} properties"
    )

In [ ]:
# Create violin plot using seaborn
if len(filtered_df) > 0:
    # Set figure size and style
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")

    # Create violin plot
    sns.violinplot(data=filtered_df, x="property_id", y="value_float", palette="Set2")

    # Customize plot
    plt.title(
        f"Extended Property Distributions - Session: {session_id}",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Property", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    plt.show()

    # Print summary statistics
    print("\nSummary Statistics:")
    summary = filtered_df.groupby("property_id")["value_float"].agg(
        ["count", "mean", "std", "min", "max"]
    )
    print(summary)

## Cleanup

In [ ]:
# Close database connection
query_backend.close()
print("✓ Connection closed")

## Summary

This notebook demonstrates:

1. **High-Level API**: Uses the same `QueryBackend` and `AnalysisContext` pattern as dashboard widgets
2. **Session Scope**: Queries property distributions aggregated across all episodes in a session
3. **Property Agnostic**: Works with any extended property (attention weights, distances, speeds, etc.)
4. **Visualization**: Creates violin plots using seaborn for distribution analysis
5. **Cloud SQL Integration**: Multiple connection approaches with security considerations

### Key Components

- `QueryBackend`: High-level database query interface
- `QueryScope.from_session()`: Define session-scope analysis
- `AnalysisContext`: Merges scope + shared parameters
- `get_property_distributions()`: Query raw property values for distributions
- `sns.violinplot()`: Visualize property distributions

### Database Connection Options (Priority Order)

**Option 1: Cloud SQL Auth Proxy + Environment Variables (RECOMMENDED)**
- Start proxy: `./cloud-sql-proxy PROJECT:REGION:INSTANCE --port 5433`
- Set environment variables to connect via localhost:5433
- Initialize `QueryBackend()` with no arguments
- **Benefits**: SSL/TLS encryption, IAM auth, no IP management
- **Best for**: Production notebooks, shared environments, secure connections
- See Cell 3 and [docs/dashboard/CLOUD_SETUP.md](../dashboard/CLOUD_SETUP.md)

**Option 2: Local DuckDB (Cell 8)**
- Initialize `QueryBackend()` with `DUCKDB_PATH` environment variable
- **Best for**: Local development, testing, offline work

**Option 3: Direct Public IP (Cells 4-6, NOT RECOMMENDED)**
- ⚠️ Bypasses proxy security features (no automatic SSL/TLS, IAM auth)
- Requires firewall configuration and authorized networks management
- Construct `PostgresConfig` with public IP and credentials
- Initialize `QueryBackend(config=db_config)`
- **Only for**: Special testing scenarios, legacy systems

### Customization

Modify these cells to customize the analysis:
- Cell 3: Configure Cloud SQL Auth Proxy connection
- Cells 4-6: Configure direct connection (if absolutely necessary)
- Cell 9/10: Select different session
- Cell 17: Change `selected_properties` to analyze different properties
- Cell 18: Customize plot style, colors, or add statistical overlays

### References

- **Recommended Setup**: [docs/dashboard/CLOUD_SETUP.md](../dashboard/CLOUD_SETUP.md)
- **Google Cloud SQL Auth Proxy**: https://cloud.google.com/sql/docs/postgres/connect-auth-proxy
- **Dashboard Widgets**: [collab_env/dashboard/widgets/](../../collab_env/dashboard/widgets/)